In [8]:
from pulp import *
import pandas as pd

# masses in dalton
phosphorus = 30.973761998
oxygen = 31.989829239
d_ribose = 150.05282342

tail_5prime = [d_ribose, oxygen, 2*oxygen + phosphorus, oxygen]

masses = pd.read_csv("../masses_all.tsv", sep="\t").set_index("nucleoside", drop=False)

In [9]:
masses

,nucleoside,monoisotopic_mass
nucleoside,,
A,A,267.09675
G,G,283.09167
C,C,243.08552
U,U,244.06954
1A,1A,281.11240
...,...,...
10G,10G,409.15970
102G,102G,425.15470
105G,105G,538.20230


In [10]:
import random
from scipy.stats import norm

nucleoside_re = re.compile(r"\d*[ACGU]")

true_sequence = nucleoside_re.findall("CG100GUU")
print(true_sequence)
n_fragments = 10

# sample random fragments from true sequence
fragment_noise = norm.rvs(scale=0.5, size=n_fragments)
def random_fragment():
    l = 0
    r = random.randint(l + 1, len(true_sequence))
    return l, r
fragments = [random_fragment() for _ in range(n_fragments)]
fragment_masses = [max(sum(masses.loc[b, "monoisotopic_mass"] for b in true_sequence[l:r]) + noise, 0.0) for noise, (l, r) in zip(fragment_noise, fragments)]
fragments, fragment_masses, fragment_noise

['C', 'G', '100G', 'U', 'U']


([(0, 2),
  (0, 4),
  (0, 2),
  (0, 5),
  (0, 2),
  (0, 1),
  (0, 1),
  (0, 2),
  (0, 1),
  (0, 2)],
 [525.8995823142466,
  1077.5630145843277,
  526.2129238753446,
  1320.8242432728673,
  526.6808051360828,
  243.07284998869528,
  242.85967183024175,
  525.6064341850912,
  242.87407749316233,
  526.4234967390169],
 array([-0.27760769,  0.22458458,  0.03573388, -0.58372673,  0.50361514,
        -0.01267001, -0.22584817, -0.57075581, -0.21144251,  0.24630674]))

In [11]:
from itertools import combinations_with_replacement
from collections import namedtuple
from itertools import groupby
from dataclasses import dataclass, field
from typing import List

min_mass_diff = 1

n = len(fragment_masses)


Candidate = namedtuple("Candidate", ["seq", "mass", "abs_diff"])

@dataclass
class SeqEst:
    i: int
    prefix_mass: float
    sorted_masses: List[float]
    max_diff: float
    candidates: List = field(default_factory=list)

    def estimate(self):
        if self.i == len(self.sorted_masses):
            yield "", 0
        else:
            mass = self.sorted_masses[self.i]
            obs_suffix_mass = mass - self.prefix_mass
            candidates = []

            # case 1: single nucleotide
            for base, abs_diff in (masses["monoisotopic_mass"] - obs_suffix_mass).abs().items():
                mass = masses.loc[base, "monoisotopic_mass"]
                candidates.append(Candidate(base, mass, abs_diff))
            # case 2: two nucleotides
            for subseq in combinations_with_replacement(masses.index, 2):
                suffix_mass = masses.loc[list(subseq), "monoisotopic_mass"].sum()
                abs_diff = abs(suffix_mass - obs_suffix_mass)
                candidates.append(Candidate(f"[{''.join(subseq)}]", suffix_mass, abs_diff))
            # case 3: zero nucleotides
            candidates.append(Candidate("", 0, abs(obs_suffix_mass)))

            best_candidates = sorted(candidates, key=lambda candidate: candidate.abs_diff)
            last_diff = None
            for suffix_mass, same_mass_candidates in groupby(best_candidates, key=lambda candidate: candidate.mass):
                same_mass_candidates = list(same_mass_candidates)
                abs_diff = same_mass_candidates[0].abs_diff
                if last_diff is not None and abs_diff - last_diff > min_mass_diff:
                    break

                for next_seq, next_diff in self.recurse(self.i + 1, self.prefix_mass + suffix_mass):
                    for candidate in same_mass_candidates:
                        self.register_result(candidate.seq + next_seq, abs_diff + next_diff)
                last_diff = abs_diff

            # case 4: more nucleotides
            # for now: do not consider
            #print(self.i, obs_suffix_mass, self.prefix_mass, self.candidates, list(self.yield_results()))
            
            yield from self.yield_results()

    def recurse(self, i, mass):
        yield from SeqEst(i, mass, self.sorted_masses, self.max_diff).estimate()

    def register_result(self, seq, diff):
        self.candidates.append((seq, diff))

    def yield_results(self):
        yield from sorted(self.candidates, key=lambda item: item[1])[:2]

def estimate_sequence(fragment_masses):
    sorted_masses = sorted(fragment_masses)
    sorted_masses = sorted_masses[:n]
    max_diff = masses["monoisotopic_mass"].min() / 2.0
    estimates = sorted(SeqEst(0, 0, sorted_masses, max_diff).estimate(), key=lambda est: est[1])[:10]
    if not estimates:
        raise ValueError("Unable to determine reasonable estimates.")
    return estimates


In [12]:
def get_diff(obs_fragment_mass, seq):
    return abs(obs_fragment_mass - masses.loc[list(seq), "monoisotopic_mass"].sum())

In [13]:
sorted_idx = sorted(range(len(fragments)), key=lambda item: fragment_masses[item])
sorted_truth = pd.Series(fragments)[sorted_idx]
sorted_fragment_masses = pd.Series(fragment_masses)[sorted_idx]
sorted_truth, sorted_fragment_masses

(6    (0, 1)
 8    (0, 1)
 5    (0, 1)
 7    (0, 2)
 0    (0, 2)
 2    (0, 2)
 9    (0, 2)
 4    (0, 2)
 1    (0, 4)
 3    (0, 5)
 dtype: object,
 6     242.859672
 8     242.874077
 5     243.072850
 7     525.606434
 0     525.899582
 2     526.212924
 9     526.423497
 4     526.680805
 1    1077.563015
 3    1320.824243
 dtype: float64)

In [14]:
estimate_sequence(fragment_masses)

[('CG[U100G]C', 2.7088577962020395), ('CG[U100G]U', 2.8922912504674514)]

In [15]:
sum([get_diff(sorted_fragment_masses.loc[12], true_sequence[:12]) for mass in sorted_fragment_masses[4:5]])

KeyError: 12

In [ ]:
true_sequence

'CGGAUCGAUCUG'

In [ ]:
list(product(masses.index, masses.index))

[('A', 'A'),
 ('A', 'G'),
 ('A', 'C'),
 ('A', 'U'),
 ('G', 'A'),
 ('G', 'G'),
 ('G', 'C'),
 ('G', 'U'),
 ('C', 'A'),
 ('C', 'G'),
 ('C', 'C'),
 ('C', 'U'),
 ('U', 'A'),
 ('U', 'G'),
 ('U', 'C'),
 ('U', 'U')]